## Package installs

After running the first package installation cell, restart the Colab runtime once (after cell has completed running), then continue. Make sure the final environment check cell completes successfully.

In [ ]:
!pip install \
  torch==2.5.1 torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121

!pip install \
  transformers==4.48.2 \
  causal-conv1d==1.5.0.post8 \
  triton==3.1.0 \
  safetensors==0.5.3 \
  huggingface_hub==0.34.5 \
  accelerate==1.10.1 \
  einops==0.8.1 \
  packaging ninja

!pip install mamba-ssm==2.2.4 --no-build-isolation

In [ ]:
!pip install evodiff

In [ ]:
!pip install pyfaidx

In [ ]:
!pip install pyBigWig

In [ ]:
import torch, transformers, causal_conv1d, triton, safetensors, huggingface_hub, accelerate, einops
print("torch:", torch.__version__, torch.version.cuda)
print("transformers:", transformers.__version__)
print("causal_conv1d:", causal_conv1d.__version__)
print("triton:", triton.__version__)
print("safetensors:", safetensors.__version__)
print("hub:", huggingface_hub.__version__)
print("accelerate:", accelerate.__version__)
print("einops:", einops.__version__)

import mamba_ssm
print("mamba_ssm:", mamba_ssm.__version__)

## Imports & tokenizer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from pyfaidx import Fasta
from transformers import AutoModel
from evodiff.utils import Tokenizer

# Define the DNA vocabulary expected by the released Gamba checkpoints.
# This avoids requiring the original training repository as an install dependency.
import sequence_models.constants as seq_constants

FIM_MIDDLE, FIM_PREFIX, FIM_SUFFIX = "_", "(", ")"
DNA_ALPHABET_PLUS = (
    seq_constants.DNA
    + "N"
    + seq_constants.MSA_PAD
    + seq_constants.MASK
    + seq_constants.STOP
    + seq_constants.START
    + FIM_MIDDLE
    + FIM_PREFIX
    + FIM_SUFFIX
)

tokenizer = Tokenizer(DNA_ALPHABET_PLUS)

print(f"Alphabet ({len(DNA_ALPHABET_PLUS)} tokens): {DNA_ALPHABET_PLUS!r}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu — consider enabling GPU Runtime'}")


## Load Genome data

In [ ]:
# Load HG38 fasta
!curl https://storage.googleapis.com/basenji_barnyard2/hg38.ml.fa.gz > hg38.ml.fa.gz

In [ ]:
# Decompress the FASTA file
!gunzip hg38.ml.fa.gz

In [ ]:
# Load the genome FASTA with pyfaidx
from pyfaidx import Fasta
genome = Fasta("hg38.ml.fa")
# pyfaidx will create an index file (.fai) on first load; this may take a few minutes.
print(f"\nFASTA loaded. Sequences: {list(genome.keys())[:8]} ...")


In [ ]:
# Get the PhyloP data
!curl https://cgl.gi.ucsc.edu/data/cactus/241-mammalian-2020v2-hub/Homo_sapiens/241-mammalian-2020v2.bigWig > 241-mammalian-2020v2.bigWig

In [ ]:
bigwig_path= "/content/241-mammalian-2020v2.bigWig"

## Load BiGamba model from HuggingFace

In [ ]:
REPO_ID = "micanonsens/bigamba-dual-step44000"
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading {REPO_ID} ...")
bigamba_model = AutoModel.from_pretrained(REPO_ID,
    trust_remote_code=True,
).eval().to(DEVICE)

print(f"Model loaded on {DEVICE}.")
print(f"Parameters: {sum(p.numel() for p in bigamba_model.parameters()):,}")

## Load ArGamba model from HuggingFace

In [ ]:
REPO_ID = "micanonsens/argamba-dual-step44000"
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

argamba_model = AutoModel.from_pretrained(REPO_ID,
    trust_remote_code=True,
).eval().to(DEVICE)

print(f"Model loaded on {DEVICE}.")
print(f"Parameters: {sum(p.numel() for p in argamba_model.parameters()):,}")

## Run predictions

In [ ]:
# Upload a BED file with regions to score.
# Required columns: chrom, start, end.
# Optional fourth column: region name.
# Coordinates should be 0-based, half-open.

from google.colab import files

print("Upload your BED file.")
uploaded = files.upload()

bed_path    = next((k for k in uploaded if k.endswith(".bed")), None)

assert bed_path,   "No BED file found — upload a .bed"

print(f"BED    : {bed_path}")

In [ ]:
# Parse uploaded BED regions
regions = []
with open(bed_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith(("#", "track", "browser")):
            continue
        parts = line.split("\t")
        chrom, start, end = parts[0], int(parts[1]), int(parts[2])
        name = parts[3] if len(parts) > 3 else f"{chrom}:{start}-{end}"
        regions.append({"chrom": chrom, "start": start, "end": end, "name": name})

print(f"Loaded {len(regions)} regions")
for r in regions[:5]:
    print(f"  {r['name']:40s}  {r['end'] - r['start']:>8,} bp")
if len(regions) > 5:
    print(f"  ... ({len(regions) - 5} more)")


In [ ]:
# Score long regions by tiling them into fixed-length model windows.
#
# ArGamba is autoregressive, so each scored base receives upstream context:
#   [upstream context | scored positions]
#
# BiGamba is bidirectional, so each scored base receives both left and right context:
#   [left context | scored positions | right context]
#
# Context-only positions are discarded to reduce edge effects.

import numpy as np
import torch
from tqdm.auto import tqdm

WINDOW_SIZE = 2048

# ArGamba / autoregressive defaults
CAUSAL_CONTEXT_SIZE = 1000
CAUSAL_PREDICT_SIZE = WINDOW_SIZE - CAUSAL_CONTEXT_SIZE  # 1048

# BiGamba / bidirectional defaults
# Keep the same number of scored bases per window for easier model comparison.
BIDIR_PREDICT_SIZE = CAUSAL_PREDICT_SIZE                 # 1048
BIDIR_TOTAL_CONTEXT = WINDOW_SIZE - BIDIR_PREDICT_SIZE   # 1000
BIDIR_LEFT_CONTEXT = BIDIR_TOTAL_CONTEXT // 2            # 500
BIDIR_RIGHT_CONTEXT = BIDIR_TOTAL_CONTEXT - BIDIR_LEFT_CONTEXT  # 500


def get_tiling_params(model_mode: str):
    """
    Return:
      left_context, right_context, predict_size

    Supported model_mode values:
      - "causal", "autoregressive", "argamba"
      - "bidirectional", "bidir", "bigamba"
    """
    mode = model_mode.lower().strip()

    if mode in {"causal", "autoregressive", "ar", "argamba"}:
        left_context = CAUSAL_CONTEXT_SIZE
        right_context = 0
        predict_size = WINDOW_SIZE - left_context

    elif mode in {"bidirectional", "bidir", "bi", "bigamba"}:
        left_context = BIDIR_LEFT_CONTEXT
        right_context = BIDIR_RIGHT_CONTEXT
        predict_size = WINDOW_SIZE - left_context - right_context

    else:
        raise ValueError(
            f"Unknown model_mode={model_mode!r}. "
            "Use 'argamba'/'causal' or 'bigamba'/'bidirectional'."
        )

    if predict_size <= 0:
        raise ValueError(
            f"Invalid tiling: WINDOW_SIZE={WINDOW_SIZE}, "
            f"left_context={left_context}, right_context={right_context}, "
            f"predict_size={predict_size}"
        )

    return left_context, right_context, predict_size


def fetch_sequence(genome, chrom, start, end):
    """Fetch sequence from pyfaidx, padding with N at boundaries."""
    chrom_len = len(genome[chrom])

    pad_left = max(0, -start)
    pad_right = max(0, end - chrom_len)

    seq_start = max(0, start)
    seq_end = min(chrom_len, end)

    seq = str(genome[chrom][seq_start:seq_end]).upper()
    return "N" * pad_left + seq + "N" * pad_right


# Build a character-to-token lookup from the model vocabulary.
char_to_idx = {c: i for i, c in enumerate(DNA_ALPHABET_PLUS)}


def tokenize(seq):
    """
    Tokenize a DNA string -> LongTensor (1, T).
    Uses one token per DNA character.
    """
    n_idx = char_to_idx.get("N", 0)
    ids = torch.tensor(
        [char_to_idx.get(c, n_idx) for c in seq],
        dtype=torch.long,
    )
    return ids.unsqueeze(0)  # (1, T)


@torch.no_grad()
def predict_window(seq, model, device=None):
    """
    Run one WINDOW_SIZE-bp sequence through the given model.

    Returns:
      1-D float32 numpy array of length WINDOW_SIZE.

    Expected model output:
      out["scaling_logits"]

    Handles:
      (1, T)
      (1, T, 1)
      (1, T, 2)  # mean/logvar; channel 0 is mean

      The conservation head may return either one value per position or a
two-channel mean/log-variance output. When two channels are present,
this function uses the predicted mean.
    """
    if device is None:
        device = next(model.parameters()).device

    ids = tokenize(seq).to(device)

    model.eval()
    out = model(ids)

    if "scaling_logits" not in out:
        raise KeyError(
            f"Model output does not contain 'scaling_logits'. "
            f"Available keys: {list(out.keys())}"
        )

    cons = out["scaling_logits"].float()

    if cons.ndim == 3:
        # Channel 0 is the predicted mean if output is (B, T, 2).
        # Also works for (B, T, 1).
        cons = cons[..., 0]

    if cons.ndim != 2:
        raise ValueError(
            f"Expected scaling_logits shape (B,T) or (B,T,C), got {tuple(cons.shape)}"
        )

    pred = cons[0].detach().cpu().numpy().astype(np.float32)

    if len(pred) != WINDOW_SIZE:
        raise ValueError(
            f"Expected {WINDOW_SIZE} predictions, got {len(pred)}. "
            "Check tokenization/model sequence length assumptions."
        )

    return pred


def predict_region(
    genome,
    chrom,
    start,
    end,
    model,
    model_mode="causal",
    model_name=None,
    device=None,
):
    """
    Sliding-window prediction over [start, end).

    ArGamba / causal:
      [left context | kept predictions]

    BiGamba / bidirectional:
      [left context | kept predictions | right context]

    Returns:
      predicted : float32 array, length = end - start
      positions : int64 array of genomic positions
    """
    if device is None:
        device = next(model.parameters()).device

    if model_name is None:
        model_name = model_mode

    left_context, right_context, predict_size = get_tiling_params(model_mode)

    region_len = end - start
    chrom_len = len(genome[chrom])
    num_windows = int(np.ceil(region_len / predict_size))

    full = np.full(region_len, np.nan, dtype=np.float32)

    desc = (
        f"{model_name} | {chrom}:{start}-{end} "
        f"[L={left_context}, keep={predict_size}, R={right_context}]"
    )

    for i in tqdm(range(num_windows), desc=desc, leave=False):
        # Genomic chunk this iteration is responsible for filling.
        chunk_start = start + i * predict_size
        chunk_end = min(chunk_start + predict_size, end)

        # Build full model window around that chunk.
        win_start = chunk_start - left_context
        win_end = chunk_start + predict_size + right_context

        # Shift to stay within chromosome bounds while preserving WINDOW_SIZE when possible.
        if win_start < 0:
            shift = -win_start
            win_start += shift
            win_end += shift

        if win_end > chrom_len:
            shift = win_end - chrom_len
            win_start -= shift
            win_end -= shift

        # Final clamp for tiny chromosomes / edge cases.
        win_start = max(0, win_start)
        win_end = min(chrom_len, win_end)

        seq = fetch_sequence(genome, chrom, win_start, win_end)

        # Pad/trim to exact WINDOW_SIZE.
        if len(seq) < WINDOW_SIZE:
            seq = seq + ("N" * (WINDOW_SIZE - len(seq)))
        elif len(seq) > WINDOW_SIZE:
            seq = seq[:WINDOW_SIZE]

        preds = predict_window(seq, model=model, device=device)

        # Effective kept segment inside the model window.
        eff_start = win_start + left_context
        eff_end = eff_start + predict_size
        eff_preds = preds[left_context : left_context + predict_size]

        # Intersect desired chunk with effective prediction span and requested region.
        take_start = max(chunk_start, eff_start, start)
        take_end = min(chunk_end, eff_end, end)

        take_len = max(0, take_end - take_start)
        if take_len == 0:
            continue

        src_offset = take_start - eff_start
        dest_offset = take_start - start

        chunk = eff_preds[src_offset : src_offset + take_len]
        full[dest_offset : dest_offset + len(chunk)] = chunk

    return full, np.arange(start, end, dtype=np.int64)

In [ ]:
MODEL_CONFIGS = {
    "ArGamba-dual": {
        "model": argamba_model,
        "mode": "argamba",
    },
    "BiGamba-dual": {
        "model": bigamba_model,
        "mode": "bigamba",
    },
}

# Move models to the selected device and set evaluation mode.
for model_name, cfg in MODEL_CONFIGS.items():
    cfg["model"].to(DEVICE)
    cfg["model"].eval()
    print(f"{model_name}: ready on {DEVICE}")


# Run predictions on all BED regions for each model.
# results[model_name][region_name] -> {chrom, positions, predicted, model_name, model_mode, label}
results = {}

for model_name, cfg in MODEL_CONFIGS.items():
    model = cfg["model"]
    model_mode = cfg["mode"]

    print(f"\n==============================")
    print(f"Running {model_name} ({model_mode})")
    print(f"==============================")

    results[model_name] = {}

    for r in regions:
        chrom, start, end, region_name = r["chrom"], r["start"], r["end"], r["name"]
        region_len = end - start

        print(f"\nPredicting {region_name} with {model_name} ({region_len:,} bp)")

        if chrom not in genome:
            print(f"  ⚠️  {chrom} not in FASTA — skipping")
            continue

        predicted, positions = predict_region(
            genome=genome,
            chrom=chrom,
            start=start,
            end=end,
            model=model,
            model_mode=model_mode,
            model_name=model_name,
            device=DEVICE,
        )

        valid = np.isfinite(predicted)
        n_valid = valid.sum()

        if n_valid > 0:
            mean = np.nanmean(predicted)
            std = np.nanstd(predicted)
            print(
                f"  {n_valid:,} / {len(predicted):,} positions  "
                f"mean={mean:.3f}  std={std:.3f}"
            )
        else:
            print(f"  ⚠️  0 / {len(predicted):,} valid positions")

        results[model_name][region_name] = {
            "chrom": chrom,
            "positions": positions,
            "predicted": predicted,
            "model_name": model_name,
            "model_mode": model_mode,
            "region_name": region_name,
            "label": f"{model_name} | {region_name}",
        }

print("\n✓ All model-region predictions done")

In [ ]:
# Load bigWig if provided
bw = None
if bigwig_path:
    try:
        import pyBigWig
        bw = pyBigWig.open(bigwig_path)
        print(f"Opened bigWig: {bigwig_path}")
    except Exception as e:
        print(f"Could not open bigWig: {e}")


def iter_prediction_results(results):
    """Yield model_name, region_name, result payload."""
    for model_name, model_results in results.items():
        for region_name, res in model_results.items():
            yield model_name, region_name, res


for model_name, region_name, res in iter_prediction_results(results):
    chrom = res["chrom"]
    positions = res["positions"]
    predicted = res["predicted"]

    model_mode = res.get("model_mode", "unknown")
    label = res.get("label", f"{model_name} | {region_name}")

    start, end = int(positions[0]), int(positions[-1]) + 1
    valid = np.isfinite(predicted)

    # Fetch true scores if bigWig provided
    true_scores = None
    if bw is not None:
        try:
            raw = bw.values(chrom, start, end)
            true_scores = np.array(raw, dtype=np.float32)
        except Exception as e:
            print(f"  ⚠️  Could not fetch true scores for {label}: {e}")

    fig, ax = plt.subplots(figsize=(14, 4))

    if true_scores is not None:
        true_valid = np.isfinite(true_scores)
        ax.scatter(
            positions[true_valid],
            true_scores[true_valid],
            s=1,
            alpha=0.4,
            color="grey",
            label="True phyloP",
            zorder=1,
        )

    ax.plot(
        positions[valid],
        predicted[valid],
        lw=1.0,
        label=f"{model_name} predicted",
        zorder=2,
    )

    ax.axhline(0, color="grey", lw=0.5, ls="--")

    # Pearson r annotation
    if true_scores is not None:
        both_valid = valid & np.isfinite(true_scores)

        if both_valid.sum() > 1:
            r = np.corrcoef(
                predicted[both_valid],
                true_scores[both_valid],
            )[0, 1]

            ax.text(
                0.02,
                0.97,
                f"Pearson r = {r:.3f}\nvalid n = {both_valid.sum():,}",
                transform=ax.transAxes,
                va="top",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.6),
            )

    ax.set_xlabel("Genomic position")
    ax.set_ylabel("phyloP")
    ax.set_title(
        f"{region_name}\n{model_name} ({model_mode}) | {chrom}:{start:,}-{end:,}"
    )
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    safe_model = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("-", "_")
    )
    safe_region = (
        region_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("-", "_")
    )

    fname = f"{safe_model}__{safe_region}_phylop.png"
    plt.savefig(fname, dpi=150)
    print(f"Saved: {fname}")

    plt.show()
    plt.close()


if bw is not None:
    bw.close()

In [ ]:
import os
import re
import zipfile
import numpy as np

# -----------------------------
# Helpers
# -----------------------------

def safe_name(x):
    """Make a filesystem/browser-track-safe name."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")


def iter_prediction_results(results):
    """
    Yield:
      model_name, region_name, res

    Supports either:
      nested results:
        results[model_name][region_name] -> payload

      or flat results:
        results[key] -> payload
    """
    for outer_key, outer_val in results.items():
        # Nested format
        if isinstance(outer_val, dict) and "predicted" not in outer_val:
            model_name = outer_key
            for region_name, res in outer_val.items():
                yield model_name, region_name, res

        # Flat format
        else:
            res = outer_val
            model_name = res.get("model_name", "model")
            region_name = res.get("region_name", outer_key)
            yield model_name, region_name, res


def grouped_results_by_model(results):
    grouped = {}

    for model_name, region_name, res in iter_prediction_results(results):
        grouped.setdefault(model_name, [])
        grouped[model_name].append((region_name, res))

    return grouped


# -----------------------------
# Optional: load true phyloP bigWig
# -----------------------------

bw = None
if bigwig_path:
    try:
        import pyBigWig
        bw = pyBigWig.open(bigwig_path)
        print(f"Opened bigWig: {bigwig_path}")
    except Exception as e:
        print(f"Could not open bigWig: {e}")


# -----------------------------
# Compute per-region correlations
# -----------------------------

correlation_rows = []

if bw is not None:
    for model_name, region_name, res in iter_prediction_results(results):
        chrom = res["chrom"]
        positions = res["positions"]
        predicted = res["predicted"]

        start = int(positions[0])
        end = int(positions[-1]) + 1

        try:
            raw = bw.values(chrom, start, end)
            true_scores = np.array(raw, dtype=np.float32)
        except Exception as e:
            print(f"⚠️ Could not fetch true scores for {model_name} | {region_name}: {e}")
            continue

        valid_pred = np.isfinite(predicted)
        valid_true = np.isfinite(true_scores)
        both_valid = valid_pred & valid_true

        valid_n = int(both_valid.sum())

        if valid_n > 1:
            r = float(np.corrcoef(predicted[both_valid], true_scores[both_valid])[0, 1])
        else:
            r = np.nan

        correlation_rows.append({
            "model_name": model_name,
            "region_name": region_name,
            "chrom": chrom,
            "start": start,
            "end": end,
            "valid_n": valid_n,
            "pearson_r": r,
        })

    print("\nPer-region correlations:")
    for row in correlation_rows:
        r_str = "nan" if not np.isfinite(row["pearson_r"]) else f"{row['pearson_r']:.4f}"
        print(
            f"  {row['model_name']} | {row['region_name']}: "
            f"r={r_str}, valid_n={row['valid_n']:,}"
        )

    print("\nAverage correlation by model:")
    for model_name in sorted(set(row["model_name"] for row in correlation_rows)):
        rows = [row for row in correlation_rows if row["model_name"] == model_name]
        rs = np.array([row["pearson_r"] for row in rows], dtype=np.float64)
        ns = np.array([row["valid_n"] for row in rows], dtype=np.float64)

        ok = np.isfinite(rs) & (ns > 1)

        if ok.sum() == 0:
            print(f"  {model_name}: no valid correlations")
            continue

        mean_r = float(np.mean(rs[ok]))
        weighted_mean_r = float(np.average(rs[ok], weights=ns[ok]))

        print(
            f"  {model_name}: "
            f"mean_r={mean_r:.4f} across {ok.sum()} regions; "
            f"weighted_mean_r={weighted_mean_r:.4f} "
            f"across {int(ns[ok].sum()):,} valid positions"
        )

else:
    print("No bigWig loaded; skipping correlation calculation.")


# -----------------------------
# Export one bedGraph per model
# -----------------------------

out_dir = "gamba_bedgraph_exports"
os.makedirs(out_dir, exist_ok=True)

bedgraph_paths = []

grouped = grouped_results_by_model(results)

for model_name, region_items in grouped.items():
    model_safe = safe_name(model_name)
    out_bg = os.path.join(out_dir, f"{model_safe}_phylop_predictions.bedGraph")

    with open(out_bg, "w") as f:
        f.write(
            f'track type=bedGraph name="{model_name} phyloP" '
            f'description="{model_name} predicted conservation scores" '
            'visibility=full autoScale=on color=31,119,180 '
            'altColor=255,127,14 priority=20\n'
        )

        n_written = 0

        for region_name, res in region_items:
            chrom = res["chrom"]
            positions = res["positions"]
            predicted = res["predicted"]

            for pos, score in zip(positions, predicted):
                if not np.isfinite(score):
                    continue

                # bedGraph is 0-based half-open: chrom start end value
                f.write(f"{chrom}\t{int(pos)}\t{int(pos) + 1}\t{float(score):.6f}\n")
                n_written += 1

    bedgraph_paths.append(out_bg)
    print(f"Saved: {out_bg} ({n_written:,} rows)")


# -----------------------------
# Save correlations table
# -----------------------------

corr_path = None

if len(correlation_rows) > 0:
    corr_path = os.path.join(out_dir, "model_region_correlations.tsv")

    with open(corr_path, "w") as f:
        f.write("model_name\tregion_name\tchrom\tstart\tend\tvalid_n\tpearson_r\n")

        for row in correlation_rows:
            f.write(
                f"{row['model_name']}\t"
                f"{row['region_name']}\t"
                f"{row['chrom']}\t"
                f"{row['start']}\t"
                f"{row['end']}\t"
                f"{row['valid_n']}\t"
                f"{row['pearson_r']}\n"
            )

    print(f"Saved: {corr_path}")


# -----------------------------
# Zip + download
# -----------------------------

zip_path = "gamba_bedgraph_exports.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in bedgraph_paths:
        z.write(path, arcname=os.path.basename(path))

    if corr_path is not None:
        z.write(corr_path, arcname=os.path.basename(corr_path))

print(f"Saved zip: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"Could not auto-download zip: {e}")


if bw is not None:
    bw.close()